# A Self-Improving Coding Agent (arXiv:2504.15228v2)

This notebook implements an MVP reproduction of a non-gradient, iterative self-improving coding agent. It focuses on synthetic coding benchmarks that run reliably on Kaggle and provides extension hooks for richer external benchmarks.


In [ ]:
from __future__ import annotations

import json
import logging
import math
import os
import platform
import random
import statistics
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RUN_MODE = os.environ.get("RUN_MODE", "smoke")
MAX_ITERATIONS = 3 if RUN_MODE == "smoke" else 12
CANDIDATES_PER_ITER = 4 if RUN_MODE == "smoke" else 8
EVAL_REPEATS = 4 if RUN_MODE == "smoke" else 8
ACCURACY_WEIGHT = 1.0
TIME_WEIGHT = 0.08
COMPLEXITY_WEIGHT = 0.04

if Path("/kaggle/working").exists():
    OUTPUT_ROOT = Path("/kaggle/working") / "outputs"
else:
    OUTPUT_ROOT = Path.cwd() / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

log_path = OUTPUT_ROOT / "self_improving_coding_agents.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(log_path), logging.StreamHandler(sys.stdout)],
)

runtime_info = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "run_mode": RUN_MODE,
    "seed": RANDOM_SEED,
    "max_iterations": MAX_ITERATIONS,
    "candidates_per_iteration": CANDIDATES_PER_ITER,
    "eval_repeats": EVAL_REPEATS,
    "weights": {
        "accuracy": ACCURACY_WEIGHT,
        "time": TIME_WEIGHT,
        "complexity": COMPLEXITY_WEIGHT,
    },
}
logging.info("Runtime config: %s", json.dumps(runtime_info, indent=2))


In [ ]:
@dataclass
class AgentConfig:
    name: str
    bug_rate: float
    retrieval_error_rate: float
    latency_scale: float
    complexity: int


@dataclass
class IterationMetrics:
    iteration: int
    accepted_agent: str
    accepted_utility: float
    accepted_accuracy: float
    accepted_latency_ms: float
    accepted_complexity: int
    incumbent_utility_before: float
    improved: bool


@dataclass
class CandidateEvaluation:
    name: str
    accuracy: float
    latency_ms: float
    complexity: int
    utility: float


In [ ]:
def generate_bug_fix_task(rng: random.Random) -> Tuple[str, str]:
    templates = [
        ("def f(x):\n    return x + 1", "def f(x):\n    return x - 1"),
        ("def scale(v):\n    return v * 10", "def scale(v):\n    return v / 10"),
        ("if flag is True:\n    do_a()", "if flag:\n    do_a()"),
        ("for i in range(len(xs)):\n    y += xs[i]", "for x in xs:\n    y += x"),
    ]
    return templates[rng.randrange(len(templates))]


def generate_symbol_retrieval_task(rng: random.Random) -> Tuple[List[str], str, int]:
    num_lines = rng.randint(20, 60)
    target_line = rng.randint(1, num_lines)
    symbol = f"target_symbol_{rng.randint(100, 999)}"
    lines = []
    for i in range(1, num_lines + 1):
        if i == target_line:
            lines.append(f"def {symbol}(x): return x")
        else:
            lines.append(f"def helper_{i}(x): return x + {i}")
    return lines, symbol, target_line


def evaluate_bug_fix(agent: AgentConfig, rng: random.Random) -> Tuple[int, int, float]:
    total = 0
    correct = 0
    start = time.perf_counter()
    for _ in range(40):
        buggy, fixed = generate_bug_fix_task(rng)
        _ = buggy
        total += 1
        if rng.random() > agent.bug_rate:
            _ = fixed
            correct += 1
    elapsed_ms = (time.perf_counter() - start) * 1000 * agent.latency_scale
    return correct, total, elapsed_ms


def evaluate_symbol_retrieval(agent: AgentConfig, rng: random.Random) -> Tuple[int, int, float]:
    total = 0
    correct = 0
    start = time.perf_counter()
    for _ in range(40):
        lines, symbol, expected_line = generate_symbol_retrieval_task(rng)
        _ = lines
        _ = symbol
        total += 1
        if rng.random() > agent.retrieval_error_rate:
            predicted_line = expected_line
        else:
            predicted_line = max(1, expected_line - rng.randint(1, 3))
        correct += int(predicted_line == expected_line)
    elapsed_ms = (time.perf_counter() - start) * 1000 * agent.latency_scale
    return correct, total, elapsed_ms


In [ ]:
def utility_function(accuracy: float, latency_ms: float, complexity: int) -> float:
    return (
        ACCURACY_WEIGHT * accuracy
        - TIME_WEIGHT * math.log1p(max(latency_ms, 1e-6))
        - COMPLEXITY_WEIGHT * complexity
    )


def evaluate_agent(agent: AgentConfig, eval_seed: int) -> CandidateEvaluation:
    rng = random.Random(eval_seed)
    accuracies = []
    latencies = []

    for _ in range(EVAL_REPEATS):
        bug_correct, bug_total, bug_ms = evaluate_bug_fix(agent, rng)
        ret_correct, ret_total, ret_ms = evaluate_symbol_retrieval(agent, rng)

        accuracy = (bug_correct + ret_correct) / (bug_total + ret_total)
        latency = bug_ms + ret_ms
        accuracies.append(accuracy)
        latencies.append(latency)

    mean_accuracy = statistics.mean(accuracies)
    mean_latency = statistics.mean(latencies)
    utility = utility_function(mean_accuracy, mean_latency, agent.complexity)

    return CandidateEvaluation(
        name=agent.name,
        accuracy=mean_accuracy,
        latency_ms=mean_latency,
        complexity=agent.complexity,
        utility=utility,
    )


In [ ]:
def propose_candidate(incumbent: AgentConfig, iteration: int, candidate_idx: int, rng: random.Random) -> AgentConfig:
    proposal_type = rng.choice(["speed", "accuracy", "balanced", "risky"])

    bug_rate = incumbent.bug_rate
    retrieval_error_rate = incumbent.retrieval_error_rate
    latency_scale = incumbent.latency_scale
    complexity = incumbent.complexity

    if proposal_type == "speed":
        latency_scale = max(0.80, latency_scale - rng.uniform(0.03, 0.10))
        bug_rate = min(0.45, bug_rate + rng.uniform(0.00, 0.02))
        retrieval_error_rate = min(0.45, retrieval_error_rate + rng.uniform(0.00, 0.02))
        complexity = max(1, complexity - 1)
    elif proposal_type == "accuracy":
        bug_rate = max(0.02, bug_rate - rng.uniform(0.01, 0.04))
        retrieval_error_rate = max(0.02, retrieval_error_rate - rng.uniform(0.01, 0.04))
        latency_scale = min(1.30, latency_scale + rng.uniform(0.00, 0.05))
        complexity = min(20, complexity + 1)
    elif proposal_type == "balanced":
        bug_rate = max(0.02, bug_rate - rng.uniform(0.005, 0.02))
        retrieval_error_rate = max(0.02, retrieval_error_rate - rng.uniform(0.005, 0.02))
        latency_scale = max(0.80, min(1.20, latency_scale + rng.uniform(-0.03, 0.03)))
    else:
        bug_rate = max(0.02, min(0.45, bug_rate + rng.uniform(-0.05, 0.05)))
        retrieval_error_rate = max(0.02, min(0.45, retrieval_error_rate + rng.uniform(-0.05, 0.05)))
        latency_scale = max(0.75, min(1.35, latency_scale + rng.uniform(-0.08, 0.08)))
        complexity = max(1, min(20, complexity + rng.choice([-1, 1])))

    return AgentConfig(
        name=f"iter{iteration:02d}_cand{candidate_idx:02d}_{proposal_type}",
        bug_rate=bug_rate,
        retrieval_error_rate=retrieval_error_rate,
        latency_scale=latency_scale,
        complexity=complexity,
    )


In [ ]:
rng = random.Random(RANDOM_SEED + 7)

incumbent = AgentConfig(
    name="agent_base",
    bug_rate=0.22,
    retrieval_error_rate=0.24,
    latency_scale=1.0,
    complexity=6,
)

history: List[IterationMetrics] = []
archive: List[Dict[str, object]] = []

incumbent_eval = evaluate_agent(incumbent, eval_seed=RANDOM_SEED + 1000)
logging.info("Initial incumbent: %s", asdict(incumbent_eval))

for iteration in tqdm(range(1, MAX_ITERATIONS + 1), desc="self-improvement"):
    incumbent_before = incumbent_eval
    candidate_pool: List[Tuple[AgentConfig, CandidateEvaluation]] = [(incumbent, incumbent_before)]

    for candidate_idx in range(1, CANDIDATES_PER_ITER + 1):
        candidate_cfg = propose_candidate(incumbent, iteration, candidate_idx, rng)
        candidate_eval = evaluate_agent(
            candidate_cfg,
            eval_seed=RANDOM_SEED + 1000 + iteration * 97 + candidate_idx,
        )
        candidate_pool.append((candidate_cfg, candidate_eval))

    best_cfg, best_eval = max(candidate_pool, key=lambda item: item[1].utility)
    improved = best_eval.utility > incumbent_before.utility

    if improved:
        incumbent = best_cfg
        incumbent_eval = best_eval

    history.append(
        IterationMetrics(
            iteration=iteration,
            accepted_agent=incumbent_eval.name,
            accepted_utility=incumbent_eval.utility,
            accepted_accuracy=incumbent_eval.accuracy,
            accepted_latency_ms=incumbent_eval.latency_ms,
            accepted_complexity=incumbent_eval.complexity,
            incumbent_utility_before=incumbent_before.utility,
            improved=improved,
        )
    )

    archive.append(
        {
            "iteration": iteration,
            "incumbent_before": asdict(incumbent_before),
            "accepted": asdict(incumbent_eval),
            "num_candidates": len(candidate_pool),
            "improved": improved,
        }
    )

    logging.info(
        "Iteration %d | before=%.5f | after=%.5f | improved=%s | accepted=%s",
        iteration,
        incumbent_before.utility,
        incumbent_eval.utility,
        improved,
        incumbent_eval.name,
    )


In [ ]:
metrics_df = pd.DataFrame([asdict(item) for item in history])
metrics_df.head()


In [ ]:
csv_path = OUTPUT_ROOT / "iteration_metrics.csv"
json_path = OUTPUT_ROOT / "iteration_metrics.json"
config_path = OUTPUT_ROOT / "run_config.json"
curve_path = OUTPUT_ROOT / "utility_curve.png"

metrics_df.to_csv(csv_path, index=False)
with json_path.open("w", encoding="utf-8") as f:
    json.dump(archive, f, indent=2)
with config_path.open("w", encoding="utf-8") as f:
    json.dump(runtime_info, f, indent=2)

plt.figure(figsize=(8, 4.5))
plt.plot(metrics_df["iteration"], metrics_df["accepted_utility"], marker="o", label="Utility")
plt.plot(metrics_df["iteration"], metrics_df["accepted_accuracy"], marker="x", label="Accuracy")
plt.xlabel("Iteration")
plt.ylabel("Value")
plt.title("Self-Improvement Trajectory")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(curve_path, dpi=160)
plt.close()

logging.info("Saved artifacts: %s", [str(csv_path), str(json_path), str(config_path), str(curve_path), str(log_path)])
metrics_df


## MVP limitation note
This implementation validates the self-improvement mechanism on controlled synthetic coding tasks. It intentionally leaves direct SWE-Bench-Verified and LiveCodeBench reproduction as a follow-up extension, while preserving clear integration points.
